In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import drive

os.chdir('/content/drive/MyDrive/tsum_24_2')
print("Updated Working Directory:", os.getcwd())

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import os
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt

# Swish Activation
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

# Squeeze-and-Excitation Block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super(SEBlock, self).__init__()
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1)
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1)

    def forward(self, x):
        scale = F.adaptive_avg_pool2d(x, 1)
        scale = F.relu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale

# MBConv Block
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio, kernel_size, stride, reduction=4):
        super(MBConv, self).__init__()
        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        layers = []
        if expand_ratio != 1:
            layers.append(nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False))
            layers.append(nn.BatchNorm2d(hidden_dim))
            layers.append(Swish())

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, stride=stride,
                      padding=kernel_size // 2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            Swish(),
            SEBlock(hidden_dim, reduction),
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        else:
            return self.block(x)

# EfficientNet
class EfficientNet(nn.Module):
    def __init__(self, num_classes=1, width_coeff=1.0, depth_coeff=1.0, dropout_rate=0.1):
        super(EfficientNet, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, int(32 * width_coeff), kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(int(32 * width_coeff)),
            Swish()
        )

        # Configuration: (in_channels, out_channels, expand_ratio, kernel_size, stride, num_blocks)
        settings = [
            (32, 16, 1, 3, 1, 1),
            (16, 24, 6, 3, 2, 2),
            (24, 40, 6, 5, 2, 2),
            (40, 80, 6, 3, 2, 3),
            (80, 112, 6, 5, 1, 3),
            (112, 192, 6, 5, 2, 4),
            (192, 320, 6, 3, 1, 1)
        ]

        self.blocks = self._make_blocks(settings, width_coeff, depth_coeff)

        self.head = nn.Sequential(
            nn.Conv2d(int(320 * width_coeff), int(1280 * width_coeff), kernel_size=1, bias=False),
            nn.BatchNorm2d(int(1280 * width_coeff)),
            Swish(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(int(1280 * width_coeff), num_classes),
            nn.Sigmoid()  # Add Sigmoid for binary classification
        )

    def _make_blocks(self, settings, width_coeff, depth_coeff):
        blocks = []
        for in_channels, out_channels, expand_ratio, kernel_size, stride, num_blocks in settings:
            in_channels = int(in_channels * width_coeff)
            out_channels = int(out_channels * width_coeff)
            num_blocks = int(num_blocks * depth_coeff)

            for i in range(num_blocks):
                blocks.append(MBConv(
                    in_channels=in_channels if i == 0 else out_channels,
                    out_channels=out_channels,
                    expand_ratio=expand_ratio,
                    kernel_size=kernel_size,
                    stride=stride if i == 0 else 1
                ))

        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

def efficientnet_b0(hidden_layers, hidden_nodes):
    return EfficientNet(width_coeff=1.0, depth_coeff=1.0)

# Training and evaluation function
def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, device, num_epochs):
    train_losses, test_losses = [], []
    train_accuracies, test_accuracies = [], []
    train_aucs, test_aucs = [], []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        all_train_labels = []
        all_train_preds = []

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]"):
            images, labels = images.to(device), labels.to(device).float()

            # Forward pass
            outputs = model(images).squeeze()

            # Compute loss
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            all_train_labels.extend(labels.cpu().numpy())
            all_train_preds.extend(outputs.detach().cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        train_accuracy = accuracy_score(all_train_labels, (np.array(all_train_preds) > 0.5).astype(int))
        train_auc = roc_auc_score(all_train_labels, all_train_preds)
        train_accuracies.append(train_accuracy)
        train_aucs.append(train_auc)

        print(f"Epoch [{epoch+1}/{num_epochs}] Training Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.4f}, AUC: {train_auc:.4f}")

        # Testing phase
        model.eval()
        test_loss = 0
        all_test_labels = []
        all_test_preds = []

        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Testing]"):
                images, labels = images.to(device), labels.to(device).float()
                outputs = model(images).squeeze()

                loss = criterion(outputs, labels)
                test_loss += loss.item()
                all_test_labels.extend(labels.cpu().numpy())
                all_test_preds.extend(outputs.detach().cpu().numpy())

        avg_test_loss = test_loss / len(test_loader)
        test_losses.append(avg_test_loss)
        test_accuracy = accuracy_score(all_test_labels, (np.array(all_test_preds) > 0.5).astype(int))
        test_auc = roc_auc_score(all_test_labels, all_test_preds)
        test_accuracies.append(test_accuracy)
        test_aucs.append(test_auc)

        print(f"Epoch [{epoch+1}/{num_epochs}] Testing Loss: {avg_test_loss:.4f}, Accuracy: {test_accuracy:.4f}, AUC: {test_auc:.4f}")

    print("Training and evaluation complete!")
    return test_aucs

In [ ]:
class BeanDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []
        for label, subdir in enumerate(['defect_bean', 'normal_bean']):
            folder = os.path.join(root_dir, subdir)
            for img_name in os.listdir(folder):
                img_path = os.path.join(folder, img_name)
                self.data.append((img_path, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = BeanDataset(root_dir="./bean_img", transform=transform)
train_data, test_data = train_test_split(dataset.data, test_size=0.2, random_state=42)

class SplitDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = SplitDataset(train_data, transform=transform)
test_dataset = SplitDataset(test_data, transform=transform)


In [ ]:
from itertools import product

def grid_search(hyperparams, model_fn, train_dataset, test_dataset, device, num_epochs=5):
    best_params = None
    best_auc = 0
    best_model = None  # 최적 모델을 저장할 변수
    results = []

    for lr, batch_size, hidden_layers, hidden_nodes in product(
        hyperparams['lr'], hyperparams['batch_size'], hyperparams['hidden_layers'], hyperparams['hidden_nodes']
    ):
        print(f"Testing combination: LR={lr}, Batch Size={batch_size}, Hidden Layers={hidden_layers}, Hidden Nodes={hidden_nodes}")
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # 모델 초기화
        model = model_fn(hidden_layers, hidden_nodes).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.BCELoss()

        # 모델 학습 및 평가
        test_aucs = train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, device, num_epochs)
        test_auc = test_aucs[-1]  # 마지막 AUC 값 사용

        # 결과 저장
        results.append((lr, batch_size, hidden_layers, hidden_nodes, test_auc))
        if test_auc > best_auc:
            best_auc = test_auc
            best_params = (lr, batch_size, hidden_layers, hidden_nodes)
            best_model = model  # 최적 모델 업데이트

    print("Best Parameters:", best_params)
    return best_params, best_model, results


# Hyperparameters to test
hyperparams = {
    'lr': [0.001, 0.0005],
    'batch_size': [32, 64],
    'hidden_layers': [1, 2],
    'hidden_nodes': [128, 256]
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_params, best_model, results = grid_search(hyperparams, efficientnet_b0, train_dataset, test_dataset, device)

# 최적 모델 구조 출력
print("Best Model Structure:")
print(best_model)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import os
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Swish Activation
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

# Squeeze-and-Excitation Block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super(SEBlock, self).__init__()
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1)
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1)

    def forward(self, x):
        scale = F.adaptive_avg_pool2d(x, 1)
        scale = F.relu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale

# MBConv Block
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio, kernel_size, stride, reduction=4):
        super(MBConv, self).__init__()
        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        layers = []
        if expand_ratio != 1:
            layers.append(nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False))
            layers.append(nn.BatchNorm2d(hidden_dim))
            layers.append(Swish())

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, stride=stride,
                      padding=kernel_size // 2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            Swish(),
            SEBlock(hidden_dim, reduction),
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        else:
            return self.block(x)

# EfficientNet
class EfficientNet(nn.Module):
    def __init__(self, num_classes=1, width_coeff=1.0, depth_coeff=1.0, dropout_rate=0.1):
        super(EfficientNet, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, int(32 * width_coeff), kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(int(32 * width_coeff)),
            Swish()
        )

        # Configuration: (in_channels, out_channels, expand_ratio, kernel_size, stride, num_blocks)
        settings = [
            (32, 16, 1, 3, 1, 1),
            (16, 24, 6, 3, 2, 2),
            (24, 40, 6, 5, 2, 2),
            (40, 80, 6, 3, 2, 3),
            (80, 112, 6, 5, 1, 3),
            (112, 192, 6, 5, 2, 4),
            (192, 320, 6, 3, 1, 1)
        ]

        self.blocks = self._make_blocks(settings, width_coeff, depth_coeff)

        self.head = nn.Sequential(
            nn.Conv2d(int(320 * width_coeff), int(1280 * width_coeff), kernel_size=1, bias=False),
            nn.BatchNorm2d(int(1280 * width_coeff)),
            Swish(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(int(1280 * width_coeff), num_classes),
            nn.Sigmoid()  # Add Sigmoid for binary classification
        )

    def _make_blocks(self, settings, width_coeff, depth_coeff):
        blocks = []
        for in_channels, out_channels, expand_ratio, kernel_size, stride, num_blocks in settings:
            in_channels = int(in_channels * width_coeff)
            out_channels = int(out_channels * width_coeff)
            num_blocks = int(num_blocks * depth_coeff)

            for i in range(num_blocks):
                blocks.append(MBConv(
                    in_channels=in_channels if i == 0 else out_channels,
                    out_channels=out_channels,
                    expand_ratio=expand_ratio,
                    kernel_size=kernel_size,
                    stride=stride if i == 0 else 1
                ))

        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Example usage
def efficientnet_b0():
    return EfficientNet(width_coeff=1.0, depth_coeff=1.0)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = efficientnet_b0().to(device)

# Dataset and DataLoader
class BeanDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []
        for label, subdir in enumerate(['defect_bean', 'normal_bean']):
            folder = os.path.join(root_dir, subdir)
            for img_name in os.listdir(folder):
                img_path = os.path.join(folder, img_name)
                self.data.append((img_path, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
        image = Image.fromarray(image)  # Convert to PIL Image for compatibility
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = BeanDataset(root_dir="./bean_img", transform=transform)

# Split dataset into train and test sets
train_data, test_data = train_test_split(dataset.data, test_size=0.2, random_state=42)

# Define separate datasets for train and test
class SplitDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = SplitDataset(train_data, transform=transform)
test_dataset = SplitDataset(test_data, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# Loss function and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training and testing loop
train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []
train_aucs, test_aucs = [], []

def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=10):
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        all_train_labels = []
        all_train_preds = []

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]"):
            images, labels = images.to(device), labels.to(device).float()

            # Forward pass
            outputs = model(images).squeeze()

            # Compute loss
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            all_train_labels.extend(labels.cpu().numpy())
            all_train_preds.extend(outputs.detach().cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        train_accuracy = accuracy_score(all_train_labels, (np.array(all_train_preds) > 0.5).astype(int))
        train_auc = roc_auc_score(all_train_labels, all_train_preds)
        train_accuracies.append(train_accuracy)
        train_aucs.append(train_auc)

        print(f"Epoch [{epoch+1}/{num_epochs}] Training Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.4f}, AUC: {train_auc:.4f}")

        # Testing phase
        model.eval()
        test_loss = 0
        all_test_labels = []
        all_test_preds = []

        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Testing]"):
                images, labels = images.to(device), labels.to(device).float()
                outputs = model(images).squeeze()

                loss = criterion(outputs, labels)
                test_loss += loss.item()
                all_test_labels.extend(labels.cpu().numpy())
                all_test_preds.extend(outputs.detach().cpu().numpy())

        avg_test_loss = test_loss / len(test_loader)
        test_losses.append(avg_test_loss)
        test_accuracy = accuracy_score(all_test_labels, (np.array(all_test_preds) > 0.5).astype(int))
        test_auc = roc_auc_score(all_test_labels, all_test_preds)
        test_accuracies.append(test_accuracy)
        test_aucs.append(test_auc)

        print(f"Epoch [{epoch+1}/{num_epochs}] Testing Loss: {avg_test_loss:.4f}, Accuracy: {test_accuracy:.4f}, AUC: {test_auc:.4f}")

    print("Training and evaluation complete!")

# Train and evaluate the model
train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=10)

# Plot results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss')
plt.plot(range(1, len(test_losses) + 1), test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, len(test_accuracies) + 1), test_accuracies, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Over Epochs')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc
import seaborn as sns

# Evaluate the model on test data
def evaluate_model(model, test_loader, device):
    model.eval()
    all_labels = []
    all_probs = []  # Store probabilities
    all_preds = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.numpy()  # Move labels to CPU for evaluation
            outputs = model(images).squeeze().cpu().numpy()  # Get probabilities
            preds = (outputs > 0.5).astype(int)  # Threshold for binary classification

            all_labels.extend(labels)
            all_probs.extend(outputs)  # Store probabilities
            all_preds.extend(preds)

    # Calculate metrics
    cm = confusion_matrix(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)
    fpr, tpr, _ = roc_curve(all_labels, all_probs)  # Use probabilities for ROC
    roc_auc = auc(fpr, tpr)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"AUC: {roc_auc:.4f}")

    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Defect', 'Normal'], yticklabels=['Defect', 'Normal'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # Plot ROC curve
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend()
    plt.show()

# Evaluate the model
evaluate_model(model, test_loader, device)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 혼동행렬 데이터
confusion_matrix = np.array([[413, 17], [72, 308]])

# 클래스 레이블
class_labels = ['Defect', 'Normal']

# 그래프 생성
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 24}, # 숫자 크기 조정
            xticklabels=class_labels, yticklabels=class_labels)

# 축 레이블 설정
plt.ylabel('True Label', fontsize=14) # y축 레이블 크기 조정
plt.xlabel('Predicted Label', fontsize=14) # x축 레이블 크기 조정
plt.title('Confusion Matrix', fontsize=16) # 제목 크기 조정

# 그래프 보이기
plt.show()
